In [1]:
import pandas as pd
import numpy as np
import os
from uuid import uuid4
from itertools import product
import concurrent.futures
from datetime import datetime
import re
from collections import namedtuple, Counter
from copy import deepcopy
from functools import cached_property

from tgsts.sequtils.trimming import trim_intersection
from utils import build_full_length_alignments, add_pipes
from tgsts.sequtils import str_to_fasta_seqrecord
from tgsts.sequtils.kmers import calculate_kmer_distance
from tgsts.sequtils.rra import identify_rrs_ali, mask_rrs
from tgsts.utils.parallel import run_concurrently
from tgsts.typeclasses import MismatchList
from tgsts.libs import ANTypingLibs
from tgsts.align import mm_profile_from_seqs
from tgsts.typeclasses import ExonDict
from tgsts.align import exonic_alignment


from sfat import Annotator 

In [2]:
output_dir = './out_reassemble_3/'
os.makedirs(output_dir, exist_ok=True)

In [3]:
path = '260709_reassemble_input.xlsx'

#df = pd.read_excel(path).replace({np.nan: None})
df = pd.read_excel(path, sheet_name=0).replace({np.nan: None})
df.head(5)

,Samples,Fragment,Library ID,Method,NumReads,Fragment Name,cds_mismatch_list,gDNA_mismatch_list,analysis_code,status_33433,Sequence
0,ARBO,1,ERAP1_BCA_1,pbAA,13,1c,"exon2:132C>T, exon2:477G>C, exon6:1165T>C",None,1,None,GAAAGTCCTGGGGGCCACCTCTAACCCACCTTCCTCCTCTACAGCA...
1,ARBO,1,ERAP1_BCA_1,pbAA,20,1g,"exon6:1134G>A, exon6:1165T>C","5utr:-268delACACACACACACACACACACGG, 5utr:-203G...",1,None,GAAAGTCCTGGGGGCCACCTCTAACCCACCTTCCTCCTCTACAGCA...
2,BM21,1,ERAP1_BCA_1,pbAA,66,1c,"exon2:132C>T, exon2:477G>C, exon6:1165T>C","5utr:-324delCAACACACACACACACACACACACACACACA, i...",1,None,GTACAGTGGCCCTTGGTAGTGCAGGAAAGTCCTGGGGGCCACCTCT...
3,BM21,1,ERAP1_BCA_1,pbAA,89,1e,"exon2:477G>C, exon6:1165T>C","5utr:-322delACACACACACACACACACACAC, 5utr:-203G...",1,None,TACAGTGGCCCTTGGTAGTGCAGGAAAGTCCTGGGGGCCACCTCTA...
4,HHKB,1,ERAP1_BCA_1,pbAA,277,1c,"exon2:132C>T, exon2:477G>C, exon6:1165T>C","5utr:-322delACACACACACACACACACACACACACACAC, in...",1,None,CAGTGGCCCTTGGTAGTGCAGGAAAGTCCTGGGGGCCACCTCTAAC...


In [4]:
#Fill rows
rows = []

prev_filled_row = None
for i, row in enumerate(df.to_dict(orient="records")):
    
    if row['Samples'] is not None:
        prev_filled_row = row
        
    for column, value in row.items():
        if 'Typing' not in column and value is None:
            row[column] = prev_filled_row[column]

        
    rows.append(row)
    

#Filter rows

#Get nonzero analysis codes and remove missing seqs
rows = [
    row 
    for row
    in rows
    if row['analysis_code'] != 0
    and row['Sequence'] is not None
]    

In [6]:
erap_reference = '...'


In [7]:
offset = -len(erap_reference.split('|')[0])
offset

-691

In [8]:
def find_occurrences(s, ch):
    return [i for i, letter in enumerate(s) if letter == ch]

def replace_char(s, i, c):
    return s[:i] + c + s[i+1:]

In [9]:
sequences_unique = {row['Sequence'].strip() for row in rows}
sequences2i = {sequence: str(i) for i, sequence in enumerate(sequences_unique)}

seqs_to_align = [
    str_to_fasta_seqrecord(
        str(i),
        sequence 
    )
    for sequence, i
    in sequences2i.items()
]



In [10]:
from tgsts.sequtils.fastio import write_fasta
from tgsts.sequtils import str_to_fasta_seqrecord
from tgsts.sequtils import clean_sequence

write_fasta(
    'erap_seqs.fasta',
    seqs_to_align,
    force=True
)

write_fasta(
    'erap_reference.fasta',
    str_to_fasta_seqrecord('ref', clean_sequence(erap_reference)),
    force=True
)



In [11]:
#Run minimap
import subprocess

subprocess.run(
    [
        "minimap2",
        "-a",
        "erap_reference.fasta",
        "erap_seqs.fasta"
    ],
    stdout=open("erap_frags_aligned.sam", "w"),
    check=True
)


[M::mm_idx_gen::0.004*2.11] collected minimizers
[M::mm_idx_gen::0.006*2.32] sorted minimizers
[M::main::0.007*2.32] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::0.007*2.20] mid_occ = 43
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::0.008*2.12] distinct minimizers: 8894 (98.97% are singletons); average occurrences: 1.025; average spacing: 5.278; total length: 48105
[M::worker_pipeline::2.303*2.94] mapped 859 sequences
[M::main] Version: 2.22-r1101
[M::main] CMD: minimap2 -a erap_reference.fasta erap_seqs.fasta
[M::main] Real time: 2.309 sec; CPU: 6.778 sec; Peak RSS: 0.227 GB


CompletedProcess(args=['minimap2', '-a', 'erap_reference.fasta', 'erap_seqs.fasta'], returncode=0)

In [12]:
#Load aligned

from utils import build_full_length_alignments

ref_aln, subread_strings = build_full_length_alignments('erap_frags_aligned.sam', 'erap_reference.fasta')

piped_ref_aln, piped_read_alns_strs = add_pipes(erap_reference, ref_aln, subread_strings)



In [13]:
class Alignment:
    def __init__(
        self,
        ref: str,
        ali: str
    ):
        if len(ref) != len(ali):
            raise ValueError(
                'Len of ali and ref does not match'
            )
            
        self.ref = ref
        self.ali = ali

        self.rrs = identify_rrs_ali(ali)
        self.seq_rr_masked = mask_rrs(clean_sequence(ali))
        self.seq_rr_masked_inner = mask_rrs(clean_sequence('|'.join(ali.split('|')[1:-1])))

    @cached_property
    def cds(self):
        """
        Return seq of only the CDS portion of the allele.
        Assumed utr, (exon, intron)... utr
        """

        only_19_exons = self.splice_base == 'T'

        ref_feats = self.ref.split('|')
        ali_feats = self.ali.split('|')
        
        cds_ali = '|'.join(ali_feats[1::2])

        only_19_exons = self.splice_base == 'T'

        if only_19_exons:
            
            #Remove exon 20
            cds_ali = '|'.join(cds_ali.split('|')[:-1])

            #Add 8 bases from intron 19
            ref_i19 = ref_feats[38]
            ali_i19 = ali_feats[38]

            bases_added = 0
            ali_iter = zip(ref_i19, ali_i19)
            ali_ext = []
            ref_ext = []
            while bases_added < 8:
                ref_base, ali_base = next(ali_iter)
                if ref_base.isalpha():
                    bases_added += 1
                ref_ext.append(ref_base)
                ali_ext.append(ali_base)

            cds_ali += ''.join(ali_ext)
                
        return clean_sequence(cds_ali)

    @cached_property
    def splice_base(self):
        """
        Get the base at 34124, which determines splicing isoform
        """

        base_count = 1
        base_index = 0

        while base_count < 34124:
            if self.ref[base_index].isalpha():
                base_count += 1
            base_index += 1

        splice_base = self.ali[base_index]

        if not splice_base in ('A', 'T', '-'):
            raise ValueError('Unexpected non AT base found at splice site')

        return self.ali[base_index]




In [14]:
erap_reference_cds = Alignment(erap_reference, erap_reference).cds

In [15]:
seq_ids = set([_.id for _ in seqs_to_align])
aligned_seq_ids = set(subread_strings.keys())

In [16]:
#Generate alignments
def generate_alignment(key, ali, trim_5utr: bool = False):
    ref = piped_ref_aln

    if trim_5utr:
        ref = '|'.join(piped_ref_aln.split('|')[:-1])
        ali = '|'.join(ali.split('|')[:-1])
        
    return Alignment(ref, ali)

#Also trim the 5_utr
args = [
    (k, ali, False)
    for k, ali
    in piped_read_alns_strs.items()
]

res = run_concurrently(
    generate_alignment,
    args,
    mode='process',
    batch_size=10,
)




Processing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 859/859 [02:29<00:00,  5.74it/s]


In [17]:
Alignment(erap_reference, erap_reference).splice_base

'T'

In [18]:
assert not res.failed

In [19]:

piped_read_alns = {
    r.args[0]: r.result
    for r
    in res.successful
}

In [20]:
help(str_to_fasta_seqrecord)

Help on function str_to_fasta_seqrecord in module tgsts.sequtils:

str_to_fasta_seqrecord(seqid: str, seq: str, description: str = '') -> Bio.SeqRecord.SeqRecord



In [21]:
def get_erap_exonic(gen_seq: str) -> str:
    ea = exonic_alignment(
        str_to_fasta_seqrecord('cons', gen_seq),
        'ERAP1',
        'erap_full'
    )
    return ExonDict(*next(iter(ea.items()))).seq

def calculate_mms(seq: str, mode: str = 'gen') -> str:
    seq = str_to_fasta_seqrecord('cons', clean_sequence(seq))
    locus_lib = ANTypingLibs().get_locus_lib('ERAP1')
    locus_lib.load_db()

    if mode == 'gen':
        mms, _ = mm_profile_from_seqs(
            seq,
            locus_lib.get_gen_ref_seq(), 
            locus_lib.get_gen_ref_seq(), 
            locus_lib,
            'gen'
        )
    elif mode == 'exon':
        mms, _ = mm_profile_from_seqs(
            seq,
            locus_lib.get_cds_ref_seq(),
            locus_lib.get_cds_ref_seq(),
            locus_lib,
            'cds'
        )
    elif mode == 'cds':
        mms, _ = mm_profile_from_seqs(
            seq,
            str_to_fasta_seqrecord('cds_ref', erap_reference_cds),
            str_to_fasta_seqrecord('cds_close', erap_reference_cds),
            locus_lib,
            'cds'
        )
        
        
    else:
        raise ValueError('mode must be gen or cds')
    mms = ', '.join([str(mm) for mm in mms])
    return mms
        
    
def get_erap_cds_prot(gen_str: str) -> str:
    annotator = Annotator('gen')
    res = annotator('ace78ee52a044de361c280b47d9e8', gen_str, fast_alignment=True)
    return res['sequence']['coding'], res['sequence']['protein']

    

In [22]:
class OverlapError(Exception):
    def __init__(
        self,
        message,
        sample,
        frags_left,
        frags_right,
        overlaps
    ):
        super().__init__(message)
        self.message = message
        self.sample = sample
        self.frags_left = frags_left
        self.frags_right = frags_right
        self.overlaps = overlaps

        self.write_output()

    def write_output(self):
        out_path = f'./{output_dir}/fails/{self.sample}.txt'

        if os.path.exists(out_path):
            os.remove(out_path)

        with open(out_path, 'w') as f:
            f.write(self.sample+ '\n')
            f.write(self.message + '\n\n')

        if_left = self.frags_left[0].fragment
        if_right = self.frags_right[0].fragment

        overlap_lines = []
        overlap_lines.append(f'         \tCoding\tNonCod\tRR    \tKmer')
        for (i, j), overlap in self.overlaps.items():
            overlap_lines.append('\t'.join([
                f'F{if_left}:{i}/F{if_right}:{j}',
                *[
                    str(int(_)).ljust(6)
                    for _
                    in overlap.match_tuple
                ]
            ]))

        with open(out_path, 'a') as f:
            f.write('\n')
            f.write('\n'.join(overlap_lines) + '\n\n')
                    

        ali_lines = []
        msa = MSA(*self.frags_left, *self.frags_right)

        #Get overlapping boundary idx
        i_start = max([
            len(re.match(r'[-|]*', ali).group())
            for ali
            in msa.alis 
        ])
        i_end = len(msa.ref) - max([
            len(re.match(r'[-|]*', ali[::-1]).group())
            for ali
            in msa.alis 
        ])

        ref = msa.ref[i_start:i_end]
        alis = [ref] + [ali[i_start:i_end] for ali in msa.alis]

        #Lose all columns which are just -s
        ref, *(alis) = [
            ''.join(ali)
            for ali
            in zip(*[
                bases
                for bases
                in zip(*alis)
                if set(bases) != {'-'}
            ])
        ]
        
        labels = [
            f'F{if_left}:1', 
            f'F{if_left}:2', 
            f'F{if_right}:1', 
            f'F{if_right}:2', 
        ]
        batch_width = 80

        for i in range(0, len(ref), batch_width):
            s = slice(i,i+batch_width)

            ref_snippet = ref[s]
            alis_snippets = [ali[s] for ali in alis]

            label_ljust = 20
            
            mm_line = ''.join([' '] * label_ljust) + ''.join([
                'v' if len(set(bases)) != 1 else ' '
                for bases
                in zip(*alis_snippets)
            ])

            ali_lines.append(mm_line)
            ali_lines.append('ref'.ljust(label_ljust) + ref_snippet)
            ali_lines.extend([
                label.ljust(label_ljust) + ali
                for label, ali
                in zip(labels, alis_snippets)
            ])

            

        with open(out_path, 'a') as f:
            f.write('\n'.join(ali_lines))
        


In [23]:
class AssemblyError(Exception):
    def __init__(self, msg: str, sample: str, overlap_lls: list['OverlapLinkedList']):
        super().__init__(msg)
        self.sample = sample

        self.write_output(msg, overlap_lls)

    def write_output(self, msg: str, overlap_lls: list['OverlapLinkedList']):
        
        out_path = f'./{output_dir}/fails/{self.sample}.txt'

        lines = []

        lines.append('SAMPLE:')
        lines.append(self.sample)
        lines.append('REASON:')
        lines.append(msg)

        #Left to right overlaps, i.e 4 frag_1/2
        l2r_overlaps = {}
        
        for overlap_ll in overlap_lls:
            for overlap in overlap_ll:
                l2r_overlaps.setdefault(
                    (overlap.frag_left.fragment, overlap.frag_right.fragment),
                    set()
                ).add(overlap)

        lines.append('\n')

        for (i_frag_left, i_frag_right), overlaps in l2r_overlaps.items():

            
            frags_left = sorted({
                overlap.frag_left
                for overlap
                in overlaps
            }, key = lambda x: id(x))
            frags_right = sorted({
                overlap.frag_right
                for overlap
                in overlaps
            }, key = lambda x: id(x))

            #Get common mismatches across all frags
            assert len(frags_left) == 2

            left_mms_common = set(frags_left[0].gen_mms) & set(frags_left[1].gen_mms)

            lines.append('▓'*79)
            lines.append(f'░░ FRAG{i_frag_left} ░░'.ljust(48, '▒').rjust(79, '▒'))
            lines.append('░'*79)
            lines.append('')
            
            for i, frag in enumerate(frags_left):
                lines.append(f'ALLELE {i+1} UNIQUE MISMATCHES:')
                lines.append('\n'.join(sorted(set(frag.gen_mms) - left_mms_common, key=mm2pos)) or 'NONE')
                lines.append('')
            lines.append('')

            
            lines.append('▓'*79)
            lines.append(f'░░ OVLP{i_frag_left}/{i_frag_right} ░░'.ljust(50, '▒').rjust(79, '▒'))
            lines.append('░'*79)
            lines.append('')

            lines.append('OVERLAP STATUS:')
            if len(overlaps) == 2:
                lines.append('UNAMBIGUOUS OVERLAP')
                lines.append('')
            else:
                overlap_alignments = [
                    overlap.get_alignment()
                    for overlap
                    in overlaps
                ]
                unique_overlap_count = len(set(overlap_alignments))
                if unique_overlap_count == 1:
                    lines.append('AMBIGUOUS HOMOZYGOUS OVERLAP')
                    lines.append('')
                else:
                    lines.append('AMBIGUOUS HETEROZYGOUS OVERLAP')
                    lines.append('')
                    
            lines.append(' '.join(['Overlap'.ljust(20), 'C', 'G', 'R', 'D']))
            for i_allele_left, frag_left in enumerate(frags_left):
                for i_allele_right, frag_right in enumerate(frags_right):

                    
                    for overlap in overlaps:
                        if (
                            overlap.frag_left is frag_left and
                            overlap.frag_right is frag_right 
                        ):
                            label_left = f'F{i_frag_left}A{i_allele_left}'
                            label_right = f'F{i_frag_right}A{i_allele_right}'
                            
                            #print(f'{label_left} / {label_right} Overlap Alignment')
                            #alignment = overlap.get_alignment(padding=20)
                            #ref, alis = alignment[0], alignment[1:]
                            #for i_ali_start in range(0, len(ref), 69):
                            #    i_ali_end  = i_ali_start + 69
                            #    print('ref'.ljust(10), ref[i_ali_start:i_ali_end], sep='')
                            #    for label, ali in zip((label_left, label_right), alis):
                            #        print(label.ljust(10), ali[i_ali_start:i_ali_end], sep='')
                            #    print()

                            lines.append(' '.join(
                                map(
                                    str,
                                    [
                                        f'{label_left}/{label_right}'.ljust(20), 
                                        *map(int, overlap.match_tuple)
                                    ]    
                                )
                            ))
            lines.append('')
            lines.append('')


        #Now do the final right frag
        right_mms_common = set(frags_right[0].gen_mms) & set(frags_right[1].gen_mms)

        lines.append('▓'*79)
        lines.append(f'░░ FRAG{i_frag_right} ░░'.ljust(48, '▒').rjust(79, '▒'))
        lines.append('░'*79)
        lines.append('')
        
        for i, frag in enumerate(frags_right):
            lines.append(f'ALLELE {i+1} UNIQUE MISMATCHES:')
            lines.append('\n'.join(sorted(set(frag.gen_mms) - right_mms_common, key=mm2pos)) or 'NONE')
            lines.append('')
        lines.append('')

        with open(out_path, 'w') as f:
            for line in lines:
                f.write(line + '\n')


In [24]:
class Fragment:
    def __init__(
        self, 
        seq: str,
        ali: Alignment,
        fragment: int,
        read_count: int,
        fragment_name: str,
        gen_mms: list[str]
    ):
        self.seq = seq
        self.ali = ali
        self.fragment = fragment
        self.read_count = read_count
        self.fragment_name = fragment_name
        self.gen_mms = gen_mms

        try:
            self.mm_pos_dict = {
                mm2pos(mm): mm
                for mm
                in gen_mms
            }
        except:
            print(gen_mms)
            raise
    
    def merge_aln(self, other: 'Fragment'):
        return MSA(self, other)

def mm2pos(mm: str):
    return int(re.search(r':(-?\d+)', mm).group(1))

mm2pos('5utr:-296delACACACACACACAC>')

-296

In [25]:
class Overlap:
    def __init__(
        self,
        frag_left: Fragment,
        frag_right: Fragment,
    ):
        self.frag_left = frag_left
        self.frag_right = frag_right

        self.match_tuple = self.assess_overlap()

    def __bool__(self):
        return self.match_tuple[0] and self.match_tuple[2]

    def __str__(self):
        return str(self.match_tuple)

    def __repr__(self):
        return f'Overlap{self.match_tuple}'

    def get_alignment(self, padding: int = 0):
        """
        Return alignment tuple of the overlap between frag left and right
        """
        overlap_aln = self.frag_left.merge_aln(self.frag_right)
        return overlap_aln.get_trimmed_alignment(padding=padding)


    def assess_overlap(self):
        overlap_aln = self.frag_left.merge_aln(self.frag_right)

        #overlap_aln.print()

        ref_ali = overlap_aln.ref
        alis = overlap_aln.alis

        i_start, i_end = overlap_aln.get_alignment_overlap_idx()

        #Get union of rr bases
        rrs1, rrs2 = overlap_aln.rrs
        rr_bases_1 = {
            i
            for rr
            in rrs1
            for i
            in range(rr['ali_start'], rr['ali_end'])
        }
        rr_bases_2 = {
            i
            for rr
            in rrs2
            for i
            in range(rr['ali_start'], rr['ali_end'])
        }
        rr_bases_both = rr_bases_1 | rr_bases_2

        #Set to false when mismatch found
        coding_match, noncoding_match, rr_match = True, True, True
        
        for i, (ref_base, *(ali_bases)) in enumerate(zip(ref_ali, zip(*alis))):
            if not i_start <= i < i_end:
                continue
            if len(set(*ali_bases)) == 1:
                continue
            #Mismtch found
            #print(i, ref_base, ali_bases, overlap_aln.get_feature(i), end='\t')
            feature = overlap_aln.get_feature(i)
            if feature == 'coding':
                coding_match = False
            elif feature == 'noncoding':
                noncoding_match = False
            else:
                raise ValueError('unexpected feature found?')

        left_tail_masked, right_head_masked = trim_intersection(
            self.frag_left.ali.seq_rr_masked,
            self.frag_right.ali.seq_rr_masked,
        )
        if len({
            str(left_tail_masked.seq),
            str(right_head_masked.seq)
        }) != 1:
            rr_match = False

        kmer_similarity = - calculate_kmer_distance(
            clean_sequence(alis[0][i_start:i_end]),
            clean_sequence(alis[1][i_start:i_end]),
            7
        )

        return coding_match, noncoding_match, rr_match, kmer_similarity

In [26]:
class MSA:
    def __init__(
        self,
        *fragments: Fragment
    ):

        if len({
            f.ali.ref
            for f
            in fragments
        }) != 1:
            raise ValueError('ref must be identical')

        self.fragments = fragments
    
        self.ref = fragments[0].ali.ref
        self.alis = [f.ali.ali for f in fragments]
        self.rrs = [f.ali.rrs for f in fragments]

    @classmethod
    def from_fragments(cls, *fragments: Fragment):
        return cls.from_alis(*[
            frag.ali
            for frag
            in fragments
        ])
            

    def get_feature(self, i):
        upstream = self.ref[:i]
        pipe_count = upstream.count('|')
        if pipe_count % 2 == 0:
            return 'noncoding'
        else:
            return 'coding'
            
    def get_alignment_overlap_idx(self):
        #Get overlapping boundary idx
        i_start = max([
            len(re.match(r'[-|]*', ali).group())
            for ali
            in self.alis
        ])
        i_end = len(self.ref) - max([
            len(re.match(r'[-|]*', ali[::-1]).group())
            for ali
            in self.alis
        ])
        return i_start, i_end

    def get_trimmed_alignment(self, padding: int = 0):
        i_start, i_end = self.get_alignment_overlap_idx()

        i_start -= padding
        i_end += padding

        return (
            self.ref[i_start:i_end],
            *[
                ali[i_start:i_end]
                for ali
                in self.alis
            ]
        )

    def consolidate(self) -> str:

        #Sort by i_fragment
        msa_sorted = MSA(*sorted(
            self.fragments,
            key = lambda x: x.fragment
        ))

        #Counts of each fragments
        fragment_counts = Counter([
            f.fragment
            for f
            in msa_sorted.fragments
        ])

        #Can only align if there is max 1 seq per fragment
        #Frags must also be contiguous
        i_fragment_prev = None
        for i_fragment, count in fragment_counts.items():
            if count > 1:
                raise ValueError(
                    'Cannot consolidate unless only 1 poss seq per fragment'
                )
            if (
                i_fragment_prev 
                and (i_fragment - i_fragment_prev) > 1
            ):
                raise ValueError(
                    'Fragments must be contiguous and overlapping'
                )
            i_fragment_prev = i_fragment
                    
        #Always choose fragment with higher read count

        #Sort fragment indexes, higher read count first
        prioritised_fragments: list[Fragment] = sorted(
            msa_sorted.fragments,
            key = lambda f: f.read_count,
            reverse=True
        )
        prioritised_alignments = [
            fragment.ali.ali
            for fragment
            in prioritised_fragments
        ]
        
        consolidated = []

        for col in range(len(prioritised_alignments[0])):
            base = '-'
            for i, aln in enumerate(prioritised_alignments):
                if aln[col] != '-':
                    base = aln[col]
                    prev_base_frag = i
                    break
            consolidated.append(base)

        return ''.join(consolidated)

    def print(self):
        ali_len = len(self.ref)
        for i in range(0, ali_len, 80):
            print(
                self.ref[i:i+80],
                *[
                    ali[i:i+80]
                    for ali
                    in self.alis
                ],
                sep='\n',
                end='\n\n'
            )




class AssembledSequence:
    def __init__(self, overlaps: list[Overlap], ambig_name: str):
        """
        Assemble all relevant information from list of frags
        """

        self.ambig_name = ambig_name
        
        frags = [
            overlaps[0].frag_left, 
            *[
                overlap.frag_right
                for overlap
                in overlaps
            ]
        ]

        msa = MSA(*frags)
        ali_str = msa.consolidate()
        
        #Remove UTRs
        ali_str_inner = '|'.join(ali_str.split('|')[1:-1])
        self.seq = clean_sequence(ali_str)
        self.seq_inner = clean_sequence(ali_str_inner)

        ali = Alignment(
            msa.ref,
            ali_str
        )

        
        self.overlaps = overlaps
        self.frags = frags
        
        self.exonic = get_erap_exonic(self.seq)
        self.cds, self.prot = get_erap_cds_prot(self.seq)
        
        self.mms = calculate_mms(self.seq_inner, mode='gen')
        self.exon_mms = calculate_mms(self.exonic, mode='exon')
        self.cds_mms = calculate_mms(self.cds, mode='cds')
        
        self.name = ''.join(
            frag.fragment_name
            for frag
            in sorted(
                frags,
                key = lambda x: x.fragment
            )
        )
        
        self.exon_count = '19' if ali.splice_base == 'T' else '20'


class CDSOnlyAssembledSequence(AssembledSequence):
    def __init__(self, overlaps: list[Overlap], ambig_name: str):
        """
        Assemble all relevant information from list of frags
        Only keep CDS alignment
        """

        super().__init__(overlaps, ambig_name)

        self.seq = 'Ambiguous'
        self.mms = 'N/A'


In [27]:
class FragmentPair(list):
    def __init__(
        self,
        frag1: Fragment,
        frag2: Fragment,
        fragment: int
    ):
        self.frag1 = frag1
        self.frag2 = frag2
        self.fragment = fragment

        super().__init__([frag1, frag2])

    def product(self, other: 'FragmentPair', enum=False):
        if not enumerate:
            for a, b in product(self, other):
                yield a, b 
        else:
            for (i, a), (j, b) in product(
                enumerate(self), enumerate(other)
            ):
                yield (i, a), (j, b)

class TilingRow:

    header = (
        'sample',
        'desc',
        'allele',
        'allele_name',
        'allele_ambig_name',
        'exon_count',
        'cds_mms',
        'cds_mms_trunc',
        'exon_mms',
        'exon_mms_trunc',
        'gen_mms',
        'gen_mms_trunc',
        'cds_homoz',
        'gen_homoz',
        '12CDS',
        '12RRA',
        '12GEN',
        '23CDS',
        '23RRA',
        '23GEN',
        '34CDS',
        '34RRA',
        '34GEN',
        '45CDS',
        '45RRA',
        '45GEN',
        'prot_seq',
        'cds_seq',
        'gen_seq',
    )
    
    def __init__(
        self,
        sample: str,
        desc: str,
        i_allele: int,
        allele_name: str = '',
        allele_ambig_name: str = '',
        exon_count: int = '',
        cds_mms: str = '',
        exon_mms: str = '',
        gen_mms: str = '',
        homozygous_cds: str = '',
        homozygous_gen: str = '',
        prot_seq: str = '',
        cds_seq: str = '',
        seq: str = '',
        overlaps: list[Overlap] = None
    ):
        overlap_dict = {}
        if overlaps:
            #Sorted overlaps for increasing frags
            overlaps = sorted(
                overlaps,
                key = lambda x: x.frag_left.fragment
            )
            for overlap in overlaps:
                label = f'{overlap.frag_left.fragment}{overlap.frag_right.fragment}'
                overlap_dict[f'{label}CDS'] = 'Y' if overlap.match_tuple[0] else 'N'
                overlap_dict[f'{label}RRA'] = 'Y' if overlap.match_tuple[2] else 'N'
                overlap_dict[f'{label}GEN'] = 'Y' if overlap.match_tuple[1] else 'N'

        cds_mms_trunc = ', '.join([
            mm
            for mm
            in cds_mms.split(', ')
            if mm and 'exon1:' not in mm
        ])
        exon_mms_trunc = ', '.join([
            mm
            for mm
            in exon_mms.split(', ')
            if mm and 'exon1:' not in mm
        ])
        if gen_mms == 'N/A':
            gen_mms_trunc = 'N/A'
            homozygous_gen = 'N/A'
        else:
            try:
                gen_mms_trunc = ', '.join([
                    mm
                    for mm
                    in gen_mms.split(', ')
                    if mm and mm2pos(mm) >= 3768
                ])
            except:
                print(type(gen_mms))
                print(gen_mms)
                raise
    
        
        self.tuple = tuple([
            sample,
            desc,
            str(i_allele),
            allele_name,
            allele_ambig_name,
            exon_count,
            cds_mms,
            cds_mms_trunc,
            exon_mms,
            exon_mms_trunc,
            gen_mms,
            gen_mms_trunc,
            homozygous_cds,
            homozygous_gen,
            overlap_dict.get('12CDS', ''),
            overlap_dict.get('12RRA', ''),
            overlap_dict.get('12GEN', ''),
            overlap_dict.get('23CDS', ''),
            overlap_dict.get('23RRA', ''),
            overlap_dict.get('23GEN', ''),
            overlap_dict.get('34CDS', ''),
            overlap_dict.get('34CDS', ''),
            overlap_dict.get('34RRA', ''),
            overlap_dict.get('45CDS', ''),
            overlap_dict.get('45RRA', ''),
            overlap_dict.get('45GEN', ''),
            prot_seq,
            cds_seq,
            seq
        ])
            
            

class FailedTiling:
    def __init__(
        self,
        sample: str,
        reason: str
    ):
        self.sample = sample
        self.reason = reason

    @property
    def rows(self):
        return [
            TilingRow(
                self.sample,
                self.reason,
                i+1,
            )
            for i 
            in range(2)
        ]
            

class Tiling:
    def __init__(
        self,
        sample: str,
        *fragment_pairs: FragmentPair
    ):
        self.sample = sample
        self.fragment_pairs = fragment_pairs
        self.assemblies = self.assemble()

    @classmethod
    def try_init(
        cls,
        sample: str,
        *args,
        **kwargs
    ):
        try:
            return cls(sample, *args, **kwargs)
        except (OverlapError, AssemblyError) as e:
            print(f'{type(e)}: {str(e)}', end='\n\n')
            return FailedTiling(sample, str(e)) 

    @property
    def rows(self):

        homozygous_cds = len({
            assembly.cds
            for assembly
            in self.assemblies
        }) == 1
        homozygous_gen = len({
            assembly.seq
            for assembly
            in self.assemblies
        }) == 1
        
        rows = [
            TilingRow(
                self.sample,
                'Gen ambig CDS unambig' if isinstance(assembly, CDSOnlyAssembledSequence) else '',
                i+1,
                assembly.name,
                assembly.ambig_name,
                assembly.exon_count,
                assembly.cds_mms,
                assembly.exon_mms,
                assembly.mms,
                'Y' if homozygous_cds else 'N',
                'Y' if homozygous_gen else 'N',
                assembly.prot,
                assembly.cds,
                assembly.seq,
                assembly.overlaps
            )
            for i, assembly
            in enumerate(self.assemblies)
        ]
        return rows

    def assemble(self):

        possible_assemblies = PossibleAssemblies(self.sample)
        
        for frags_left, frags_right in zip(
            self.fragment_pairs[:-1],
            self.fragment_pairs[1:],
        ):
            i_f_left: int = frags_left.fragment 
            i_f_right: int = frags_right.fragment 
            overlaps = {}
            for (i, frag_left), (j, frag_right) in frags_left.product(
                frags_right, 
                enum=True
            ):
                overlap = Overlap(frag_left, frag_right)
                #print(f'- A{i}/A{j}', overlap.match_tuple, bool(overlap))
                overlaps[(i,j)] = overlap

            acceptable_overlaps = {
                k:v
                for k, v
                in overlaps.items()
                if v
            }

            if len(acceptable_overlaps) == 0:
                raise OverlapError(
                    f'No overlaps found between F{i_f_left} and F{i_f_right}',
                    self.sample,
                    frags_left,
                    frags_right,
                    overlaps
                )
            if {_[0] for _ in acceptable_overlaps.keys()} != {0, 1}:
                raise OverlapError(
                    f'F{i_f_left} could not overlap both frags to F{i_f_right}',
                    self.sample,
                    frags_left,
                    frags_right,
                    overlaps
                )
            if {_[1] for _ in acceptable_overlaps.keys()} != {0, 1}:
                raise OverlapError(
                    f'F{i_f_left} could not overlap to both F{i_f_right} frags',
                    self.sample,
                    frags_left,
                    frags_right,
                    overlaps
                )

            #If there are three possible overlaps, one is always
            #removeable as one fragment has an unambiguous choicfe
            if len(acceptable_overlaps) == 3:
                f_left_counts = {0: [], 1:[]}
                for _i, _j in acceptable_overlaps.keys():
                    f_left_counts[_i].append(_j)
                #Find the fragment which has two options
                greedy_frag = 0 if len(f_left_counts[0]) == 2 else 1
                modest_frag = int(not greedy_frag)

                del acceptable_overlaps[(
                    greedy_frag, 
                    f_left_counts[modest_frag][0]
                )]
            #If there are 2 possible overlaps, perfect and unambig
            #If there are 4 ambigs, this is am ambiguous overlap.
            #This may or may not be a problem, depending on whether there
            #are polymorphisms both upstream and downstream that become
            #ambiguous due to the overlap ambiguity in this frag
            #Either way, here we add all possibilities
            

            possible_assemblies.update(list(acceptable_overlaps.values()))
            
        #Get chosen overlaps
        try:
            overlaps1, overlaps2, ambig_name1, ambig_name2 = possible_assemblies.get_accepted_overlaps(
                accept_intronic_ambiguity=False
            )
            #Consolidate final sequence from the frags 
            assembled1 = AssembledSequence(overlaps1, ambig_name1)
            assembled2 = AssembledSequence(overlaps2, ambig_name2)
        except AssemblyError as ae:
            overlaps1, overlaps2, ambig_name1, ambig_name2 = possible_assemblies.get_accepted_overlaps(
                accept_intronic_ambiguity=True
            )
            assembled1 = CDSOnlyAssembledSequence(overlaps1, ambig_name1)
            assembled2 = CDSOnlyAssembledSequence(overlaps2, ambig_name2)
            
        

        return assembled1, assembled2

            
class OverlapLinkedList:
    def __init__(
        self,
        head: Overlap,
        tail: 'OverlapLinkedList' = None
    ):
        self.head = head
        self.tail = tail

    def update(
        self,
        new_head: Overlap
    ) -> 'OverlapLinkedList':
        if self.head.frag_right is not new_head.frag_left:
            raise ValueError('New head doesnt match list')
        return OverlapLinkedList(
            new_head,
            self
        )

    def __iter__(self) -> list[Overlap]:
        '''
        return list of overlaps corresponding to the tree 
        '''
        
        frag_list = [self.head]
        
        if self.tail is not None:
            frag_list.extend(list(self.tail)[::-1])
            
        return iter(frag_list[::-1])

class PossibleAssemblies(list):
    def __init__(self, sample: str):
        self.sample = sample
        self.overlap_lls = []
        super().__init__(self.overlap_lls)

    def update(self, new_overlaps: list[Overlap]):

        #If no present linked lists, create them
        if not self.overlap_lls:
            for overlap in new_overlaps:
                self.overlap_lls.append(OverlapLinkedList(overlap))
            super().__init__(self.overlap_lls)
        #Else, assign them
        else:
            new_overlap_lls = []
            for ll, overlap in product(self.overlap_lls, new_overlaps): 
                if ll.head.frag_right is overlap.frag_left:
                    new_overlap_lls.append(ll.update(overlap))

            self.overlap_lls = new_overlap_lls 
            super().__init__(self.overlap_lls)

    def get_accepted_overlaps(self, accept_intronic_ambiguity: bool = False):

        unique_sequences = {}
        
        for ll in self.overlap_lls:
            overlaps = list(ll)

            frags = [
                overlaps[0].frag_left, 
                *[
                    overlap.frag_right
                    for overlap
                    in overlaps
                ]
            ]

            if not accept_intronic_ambiguity:
                unique_sequences.setdefault(tuple(
                    f.ali.seq_rr_masked_inner
                    for f
                    in frags
                ), []).append(overlaps)
            else:
                unique_sequences.setdefault(tuple(
                    f.ali.cds
                    for f
                    in frags
                ), []).append(overlaps)
                

        if len(unique_sequences) > 2:
            raise AssemblyError(
                'More than 2 possible sequences - unable to tile across frags',
                self.sample,
                self.overlap_lls
            )
        elif len(unique_sequences) == 1:
            possibilities = next(iter(unique_sequences.values()))
            chosen_overlaps1, chosen_overlaps2, ambig_name1, ambig_name2 = self.select_two_alleles(
                possibilities,
                possibilities
            )
        elif len(unique_sequences) == 2:
            possibilities = list(unique_sequences.values())
            chosen_overlaps1, chosen_overlaps2, ambig_name1, ambig_name2 = self.select_two_alleles(
                *possibilities
            )
        else: 
            raise ValueError('No accepted overlaps? Code logic gone wrong?')

        return chosen_overlaps1, chosen_overlaps2, ambig_name1, ambig_name2

    def select_two_alleles(
        self,
        poss1: list[list[Overlap]], 
        poss2: list[list[Overlap]]
    ) -> tuple[list[Overlap], list[Overlap]]:

        acceptable_pair_scores = {}
        #Find best combination of overlaps that only consumes each frag once
        for (i, overlaps1), (j, overlaps2) in product(
            enumerate(poss1), 
            enumerate(poss2)
        ):
            #Skip if any overlapping fragments
            fragment_ids_1 = {
                id(frag)
                for overlap
                in overlaps1
                for frag
                in (overlap.frag_left, overlap.frag_right)
            }
            fragment_ids_2 = {
                id(frag)
                for overlap
                in overlaps2
                for frag
                in (overlap.frag_left, overlap.frag_right)
            }
            if fragment_ids_1 & fragment_ids_2:
                continue
            key = tuple(sorted(
                [tuple(overlaps1), tuple(overlaps2)],
                key = lambda x: self.overlaps_to_name(x)
            ))
            acceptable_pair_scores[key] = sum([
                overlap.match_tuple[3] #Sort on kmer distance
                for overlaps
                in (overlaps1, overlaps2)
                for overlap
                in overlaps
            ])

        #Get selected overlaos
        chosen_overlaps1, chosen_overlaps2 = sorted(
            acceptable_pair_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[0][0]

        chosen_name1 = self.overlaps_to_name(chosen_overlaps1) 
        chosen_name2 = self.overlaps_to_name(chosen_overlaps2) 

        possible_names1, possible_names2 = set(), set()
        for possible_overlaps1, possible_overlaps2 in acceptable_pair_scores.keys():
            possible_name1 = self.overlaps_to_name(possible_overlaps1)
            possible_name2 = self.overlaps_to_name(possible_overlaps2)
            possible_names1.add(possible_name1)
            possible_names2.add(possible_name2)

        
        ambig_name1 = '/'.join(sorted(
            possible_names1,
            key = lambda x: (not x == chosen_name1, x)
        ))
        ambig_name2 = '/'.join(sorted(
            possible_names2,
            key = lambda x: (not x == chosen_name2, x)
        ))
        
        return chosen_overlaps1, chosen_overlaps2, ambig_name1, ambig_name2
        

    def overlaps_to_name(
        self,
        overlaps: list[Overlap]
    ) -> list[str]:
        frags = [
            overlaps[0].frag_left, 
            *[
                overlap.frag_right
                for overlap
                in overlaps
            ]
        ]
        name = ''.join(
            frag.fragment_name
            for frag
            in sorted(
                frags,
                key = lambda x: x.fragment
            )
        )
        return name
        

In [28]:
-True

-1

In [29]:
directory = f"{output_dir}/fails"

# Creates directory and intermediate folders if they don't exist
os.makedirs(directory, exist_ok=True)

In [30]:
# Load into sample dict
samples = {}

for row in rows:
    sample = samples.setdefault(row['Samples'], {})

    sequence = row['Sequence'].strip()
    alignment = piped_read_alns[sequences2i[sequence]]
    i_fragment = row['Fragment']
    read_count = int(row['NumReads'])
    fragment_name = row['Fragment Name']
    gen_mms = [
        mm
        for mm
        in (row['gDNA_mismatch_list'].split(', ') if row['gDNA_mismatch_list'] else [])
        if mm != 'No_MM'
    ]

    fragment = Fragment(
        seq=sequence,
        ali=alignment,
        fragment=i_fragment,
        read_count=read_count,
        fragment_name=fragment_name,
        gen_mms=gen_mms
    )

    if row['analysis_code'] == 1:
        sample.setdefault(i_fragment, []).append(fragment)
    elif row['analysis_code'] == 2:
        sample.setdefault(i_fragment, []).extend([fragment, deepcopy(fragment)])
    elif row['analysis_code'] == None:
        continue
    else:
        continue
        raise Exception('Other code found:', row['Typing Results::analysis_code'])
        
    #print(len(sample.get(row['Typing Results::Fragment'], None)))
    
#Lose weird numbered samples
for sample_id, fragment_info in [*samples.items()]:
    for i in range(1, 6):
    #for fragment, sequences in fragment_info.items():
        if len(fragment_info.get(i, [])) != 2:
            print('Deleting', sample_id)
            print(sample_id.ljust(20), f'fragment {i}', f'{len(fragment_info.get(i, []))} sequences', sep='\t')
            del samples[sample_id]
            break

    
#Lose None sequence
for sample_id, fragment_info in [*samples.items()]:
    for fragment, sequences in fragment_info.items():
        if None in sequences:
            print('Deleting', sample_id)
            print(
                sample_id.ljust(20), 
                f'fragment {fragment}', 
                f'{len(sequences)} sequences', 
                sep='\t'
            )
            del samples[sample_id]
            break

In [31]:
print(samples.keys())

for k in list(samples.keys()):
    if '/' in k:
        samples[k.replace('/', '-')] = samples[k]
        del samples[k]

        
print(samples.keys())

dict_keys(['ARBO', 'BM21', 'HHKB', 'IBW9', 'LKT3', 'MADURA', 'DHIF/DHF', 'EHM', 'JVM', 'MANIKA', 'TISI', 'WT100BIS', 'DAUDI', 'BH', 'BOLETH', 'LZL', 'SPO010', 'EK', 'HARA', 'ISH3', 'DEU', 'LUY', 'DO208915', 'CB', 'HAG', 'DAN723', 'MZ070782', 'HO104', 'HO301', 'KHAGNI', 'BM9', 'JESTHOM', 'WT8', 'QBL', 'BSM', 'JHAF', 'EJ32B', 'AM/AMR', 'WT49', 'VAVY', 'AMAI', 'STEINLIN', 'CB6B', 'HOR', 'KPE', 'SCHU', 'TAB089', 'WIN', 'MT14B', 'WT51', 'WDV', 'SWEIG007', 'J0528239', 'PLH', 'HOM-2', 'DBB', 'DK1', 'JS', 'LBF/LBUF', 'MGAR', 'PITOUT', 'RSH', 'TOKUNAGA', 'APA', 'FH13', 'L0081785', 'M7', 'PETCH', 'AZH', 'BUR, E', 'CALOGERO', 'FH06', 'GEE018', 'KOSE', 'KY', 'LCL721', 'OZB', 'RML', 'WEWAK1', 'PGF', 'BEI', 'ELON', 'BM92', 'JAP-NF', 'KAS011', 'MOU', 'BOB', 'DUCAF', 'EA', 'LKT14', 'LWAGS', 'OMW', 'PMG075', 'WJR076', 'BTB', 'E4181324', 'JBUSH', 'PE117', 'PF97387', 'WT24', 'YAR', 'AWELLS', 'BM16', 'BER', 'BM14', 'CF996', 'COX', 'DKB', 'HID', 'KT12', 'LD2B', 'MLF/MCF', 'SLE005', 'SPACH/SPL', 'T7526', 'T

In [32]:
sample_fragment_pairs = {}

for sample_id in samples:
    sample_fragment_pairs[sample_id] = []
    for i_fragment, fragments in samples[sample_id].items():
        fragment_pair = FragmentPair(*fragments, i_fragment)
        sample_fragment_pairs[sample_id].append(fragment_pair)
    sample_fragment_pairs[sample_id].sort(
        key = lambda x: x.fragment
    )

In [ ]:
tilings = []

for sample_id, fragment_pairs in sample_fragment_pairs.items():

    if sample_id !='DAN723':
        continue
        
    print(sample_id)

    for fragments in fragment_pairs:
        for fragment in fragments:
            print(fragment.fragment_name, end='\t')
        print()
    print()
    
    tiling = Tiling.try_init(sample_id, *fragment_pairs)
    
    tilings.append(tiling)
    print(tiling.rows)
    print('\n\n\n')

    break

for row in tiling.rows:
    for h, x in zip(row.header, row.tuple):
        print(h)
        print(x)
        print('___')
    print('\n\n')

In [34]:
from tgsts.utils.parallel import run_concurrently

concurrent_results = run_concurrently(
    Tiling.try_init,
    [
        tuple([sample_id, *fragment_pairs])
        for sample_id, fragment_pairs
        in sample_fragment_pairs.items()
    ],
    mode = 'process',
    raise_exceptions = False,
    max_workers = 30,
)

Processing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 167/167 [15:03<00:00,  5.41s/it]


In [35]:
len(concurrent_results.successful), len(concurrent_results.failed)

if concurrent_results.failed:
    raise concurrent_results.failed[0].exception

In [36]:
tilings = [
    result.result
    for result
    in concurrent_results.successful
]
    

In [37]:
output_path = f'{output_dir}/assembled.csv'

import csv

with open(output_path, 'w', newline='\n') as csvfile:
    writer = csv.writer(
        csvfile, 
        delimiter=',',
        quotechar='"', 
        quoting=csv.QUOTE_MINIMAL
    )
    writer.writerow(TilingRow.header)
    for tiling in tilings:
        if isinstance(tiling, Tiling):
            for row in tiling.rows:
                writer.writerow(row.tuple)






In [38]:
output_path = f'{output_dir}/unassembled.csv'

import csv

with open(output_path, 'w', newline='\n') as csvfile:
    writer = csv.writer(
        csvfile, 
        delimiter=',',
        quotechar='"', 
        quoting=csv.QUOTE_MINIMAL
    )
    writer.writerow(TilingRow.header)
    for tiling in tilings:
        if not isinstance(tiling, Tiling):
            for row in tiling.rows:
                writer.writerow(row.tuple)



